In [1]:
import nest_asyncio
nest_asyncio.apply()

import sys
sys.path.append("../")

from dotenv import load_dotenv

assert load_dotenv("../.env", override=True), "failed to load .env"

In [ ]:
# !playwright install --with-deps chromium
!docker exec crystalvision-ollama-1 ollama pull $OLLAMA_CODE_MODEL

In [ ]:
import os
# from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from qdrant_client import QdrantClient
from crystalvision.lang.embeddings import FastEmbedEmbeddingsGPU

llm = ChatOllama(model=os.getenv("OLLAMA_CODE_MODEL"), temperature=0)
# embedding = OllamaEmbeddings(model=os.getenv("OLLAMA_EMBED_MODEL"))
embedding = FastEmbedEmbeddingsGPU(model_name=os.getenv("FASTEMBED_TEXT_MODEL"), parallel=0)

qclient = QdrantClient(prefer_grpc=True)
# qclient.set_model(embedding_model_name=os.getenv("FASTEMBED_TEXT_MODEL"))

In [ ]:
from crystalvision.lang.loaders import explain_database

df = explain_database()
df.head(3)

In [5]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
df["text_en"].sample(20)

In [ ]:
df[df['text_en'].str.contains('《LB》')].sample(1)[['name_en', 'code', 'cost', 'element', 'text_en']]

In [ ]:
df[df['text_en'].str.contains('br')].sample(1)[['name_en', 'code', 'cost', 'element', 'text_en']]

In [9]:
from langchain_qdrant import QdrantVectorStore

vectorstore = None
if qclient.collection_exists(collection_name="test_collection_gpu"):
    vectorstore = QdrantVectorStore(client=qclient, collection_name="test_collection_gpu", embedding=embedding)
    
vectorstore

In [10]:
from crystalvision.lang.docs import gather_documents

if vectorstore is None:
    documents = []

    for document in gather_documents():
        async for doc in document.alazy_load():
            documents.append(doc)

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import models as qmodels


if not qclient.collection_exists(collection_name="test_collection_gpu"):
    distance = qmodels.Distance.COSINE
    embedding_vector = embedding.embed_query("Getting the dimesionality!")
    qclient.create_collection(
        collection_name="test_collection_gpu",
        vectors_config=qmodels.VectorParams(
            size=len(embedding_vector),
            distance=distance
        ),
    )

if qclient.count("test_collection_gpu").count < 1:
    vectorstore = QdrantVectorStore(client=qclient, collection_name="test_collection_gpu", embedding=embedding)
    await vectorstore.aadd_documents(
        documents=documents,
    )


# Retrieve and generate using the relevant snippets of the blog.
retriever = vectorstore.as_retriever(
    # search_type="mmr",
    # search_kwargs={'lambda_mult': 1.0}
    # search_type="similarity_score_threshold",
    # search_kwargs={'score_threshold': 0.35}
)
retriever

In [ ]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "rules_search",
    "Search for rules information about individual cards or the rules of FFTCG. For any questions about FFTCG rules, you must use this tool after looking up the relevant cards! You should search for the mechanic, code, and/or card name to find the rules.",
)

prefix = "You are a pandas agent. You must work with the DataFrame df where each row represents a unique card. "
for col in df.columns:
    if (col_desc := df[col].attrs.get("description", "")):
        prefix += f"'{col}' refers to {col_desc}. "

prefix += "When you refer to a card, so in a a NAME [CODE] format. Please note paying a Crystal ([C]) is different resource from paying CP, the card name Lightning is different from the element Lightning. You can never overwrite or override the dataframe or df. You can never load or read a csv. You must never make up information."

print(prefix)

agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    include_df_in_prompt=None,
    allow_dangerous_code=True,
    prefix=prefix,
    extra_tools=[retriever_tool]
)

In [ ]:
agent.invoke("How many Auron cards are there?")

In [ ]:
agent.invoke("Are all Auron cards mono element?")

In [ ]:
agent.invoke("Tell me about an existing random multi element card.")

In [ ]:
agent.invoke("Tell me about a card that consists of 3 or more elements.")

In [ ]:
agent.invoke("I want an Ice element card that has the highest power.")

In [ ]:
agent.invoke("Tell me about 1-001H.")

In [ ]:
agent.invoke("What are the codes for cards named 'Auron'?")

In [ ]:
agent.invoke("What does each 'Auron' do?")

In [ ]:
agent.invoke("I need a chart showing the distribution of elements")

In [ ]:
output = agent.invoke("I want the lowest power forward that also has an ex burst.")["output"]
print(output)
agent.invoke(output + " What is its EX ability (ex abilities can be parsed from text_en)?")

In [ ]:
output = agent.invoke("Pick a card at random whose type is backup but also has an ex burst.")["output"]
print(output)
agent.invoke(output + " What is its EX ability (ex abilities can be parsed from text_en)?")

In [ ]:
agent.invoke("Tell me about paying crystal(s) and the Ice element card named Lady Lilith.")

In [ ]:
agent.invoke("Tell me about the card with code/serial 19-138S.")

In [ ]:
agent.invoke("While controlling 19-138S Lightning, what happens if I party attack with exactly 2 Category XIII forwards?")

In [ ]:
agent.invoke("While controlling 19-138S Lightning, what happens if I party attack with exactly 2 Category XII forwards?")

In [ ]:
agent.invoke("Tell me about paying crystal(s) for the ability on the card 23-038H. What is the rule?")

In [ ]:
agent.invoke("Tell me the rules about paying crystal(s) and the card 23-038H.")

In [ ]:
agent.invoke("If I play Lasswell 16-042R on a field with no other characters present, does it have to dull and freeze itself?")

In [ ]:
agent.invoke("Tell me about a random card with a '《ダル》' ability in its text_en. I want its name, code, cost, element, and exact text.")